# Reference unfragmented CCSD(T) geometry optimization

This notebook performs a **reference geometry optimization of propylene at the unfragmented (full-molecule) CCSD(T) level** with analytic nuclear gradients. It serves as the benchmark against which the results of the EWF simulations in this repository are compared: because no fragmentation or embedding is involved, the optimized geometry obtained here is free of the approximations introduced by the EWF density assembly, and any deviation of an EWF-optimized structure from this one measures the error of the embedding treatment itself.

The optimized geometry produced at the end of this notebook is the `propylene_ccsd_t.txt` reference used by [`../Geom_Comparison_Tool/`](../Geom_Comparison_Tool/) to compute RMSD and per-atom deviations of the `rdm_t` and `rdm_t_lambda` EWF geometries.

**Settings are chosen to match the EWF runs** (`../source/config.yaml`): the same input geometry (`propylene.txt`), the same STO-3G basis, charge 0, singlet, no symmetry — so the comparison isolates the effect of fragmentation, not of the basis or reference.

**Workflow:**
1. Read the starting geometry and build the PySCF molecule.
2. Set up the RHF mean-field reference.
3. Define a CCSD(T) energy + analytic gradient scanner.
4. Optimize the geometry with geomeTRIC.
5. Print the converged reference geometry.

## 1. Input geometry and molecule setup

Read the starting propylene structure from `propylene.txt` (plain `Element x y z` format, Angstrom) and build the `gto.Mole` object.

In [1]:
from pyscf import gto

def read_geometry(fname):
    f = list(open(fname,'r').readlines())
    f = [fi.split() for fi in f]
    f = [[fi[0],(float(fi[1]),float(fi[2]),float(fi[3]))] for fi in f]
    return f

# ---------- Your structure

geo = read_geometry('propylene.txt')
mol = gto.Mole()
mol.build(     
    atom       = geo,
    basis      = 'sto-3g',
    verbose    = 0,
    charge     = 0,
    spin       = 0,
    symmetry   = False)

## 2. Mean-field reference (RHF)

Restricted Hartree–Fock provides the reference determinant for the coupled-cluster treatment. The actual SCF at each optimization step is re-run inside the scanner below; this cell just establishes the mean-field object for the initial geometry.

In [2]:
from pyscf import scf
mf = scf.RHF(mol)

## 3. CCSD(T) energy + analytic gradient scanner

`ccsd_t_scanner` evaluates, at each candidate geometry supplied by the optimizer:

1. **RHF** — re-converge the mean field.
2. **CCSD** — solve the coupled-cluster ground state.
3. **(T)** — add the perturbative-triples energy correction (`ccsd_t()`).
4. **Λ equations** — solve the CCSD(T) lambda equations (`ccsd_t_lambda`), which provide the relaxed density required for analytic gradients.
5. **Analytic nuclear gradient** — contract the relaxed densities with the integral derivatives (`pyscf.grad.ccsd_t`).

Note the contrast with the EWF workflow: here the Λ equations are solved exactly for the full molecule, so the gradient is the exact derivative of the CCSD(T) energy — there is no fragmentation, no density assembly, and therefore no missing density-response term (see the discussion in the repository [`README.md`](../README.md)).

`as_pyscf_method` wraps the scanner so PySCF's geometry-optimizer interface can drive it.

In [3]:
from pyscf import cc
from pyscf.cc import ccsd_t_lambda_slow as ccsd_t_lambda
from pyscf.grad import ccsd_t as ccsd_t_grad
from pyscf.geomopt.addons import as_pyscf_method

#Build the energy and gradient scanner function
def ccsd_t_scanner(mol_input):
    """
    Computes both the total CCSD(T) energy and nuclear gradients
    for a given molecular configuration.
    """
    # Mean-Field Hartree-Fock
    mf = scf.RHF(mol_input).run()
    
    # Ground-state CCSD
    mycc = cc.CCSD(mf).run()
    
    # Calculate energy correction for perturbative triples (T)
    et = mycc.ccsd_t()
    e_tot = mycc.e_tot + et
    
    # Resolve the CCSD(T) Lambda equations required for gradients
    eris = mycc.ao2mo()
    conv, l1, l2 = ccsd_t_lambda.kernel(mycc, eris, mycc.t1, mycc.t2)
    
    # Compute the analytical nuclear gradients
    de = ccsd_t_grad.Gradients(mycc).kernel(mycc.t1, mycc.t2, l1, l2, eris=eris)
    
    return e_tot, de

#Adapt the scanner to make it compatible with PySCF geometry optimizers
pyscf_method_wrapper = as_pyscf_method(mol, ccsd_t_scanner)

## 4. Geometry optimization (geomeTRIC)

Run the optimization with PySCF's geomeTRIC interface. The convergence criteria listed below are geomeTRIC's defaults (Gaussian-style thresholds), written out explicitly for transparency. Because the CCSD(T) gradient is exact, the optimizer can be held to these **tight criteria** — unlike the EWF optimizations, which currently require looser thresholds to accommodate the residual gradient floor of the embedding.

In [4]:
# geometric
from pyscf.geomopt.geometric_solver import optimize

conv_params = { # These are the default settings
    'convergence_energy': 1e-6,  # Eh
    'convergence_grms': 3e-4,    # Eh/Bohr
    'convergence_gmax': 4.5e-4,  # Eh/Bohr
    'convergence_drms': 1.2e-3,  # Angstrom
    'convergence_dmax': 1.8e-3,  # Angstrom
}
mol_eq = optimize(pyscf_method_wrapper, **conv_params)

geometric-optimize called with the following command line:
/opt/homebrew/Caskroom/miniforge/base/envs/classical/lib/python3.12/site-packages/ipykernel_launcher.py --f=/Users/kaliakd/Library/Jupyter/runtime/kernel-v309bfb6d9538a642b96feaa149badadccf78fc1ee.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%

cycle = 1  norm(lambda1,lambda2) = 0.0151424
cycle = 2  norm(lambda1,lambda2) = 0.00335302
cycle = 3  norm(lambda1,lambda2) = 0.00138561
cycle = 4  norm(lambda1,lambda2) = 0.000324528
cycle = 5  norm(lambda1,lambda2) = 0.000107761
cycle = 6  norm(lambda1,lambda2) = 2.51266e-05
cycle = 7  norm(lambda1,lambda2) = 6.7572e-06
cycle = 8  norm(lambda1,lambda2) = 2.30548e-06
cycle = 9  norm(lambda1,lambda2) = 8.87937e-07
cycle = 10  norm(lambda1,lambda2) = 1.76897e-07
cycle = 11  norm(lambda1,lambda2) = 5.95737e-08
cycle = 12  norm(lambda1,lambda2) = 3.89721e-08
cycle = 13  norm(lambda1,lambda2) = 3.06389e-08
cycle = 14  norm(lambda1,lambda2) = 1.62057e-08
cycle = 15  norm(lambda1,lambda2) = 1.29202e-08
cycle = 16  norm(lambda1,lambda2) = 1.1572e-08
cycle = 17  norm(lambda1,lambda2) = 1.3725e-08
cycle = 18  norm(lambda1,lambda2) = 1.04433e-08
cycle = 19  norm(lambda1,lambda2) = 8.25864e-09


Step    0 : Gradient = 4.482e-02/5.095e-02 (rms/max) Energy = -115.8591865223
Hessian Eigenvalues: 2.30000e-02 2.30000e-02 2.54027e-02 ... 2.58307e-01 2.58467e-01 3.99459e-01


cycle = 1  norm(lambda1,lambda2) = 0.0103725
cycle = 2  norm(lambda1,lambda2) = 0.00251652
cycle = 3  norm(lambda1,lambda2) = 0.00100632
cycle = 4  norm(lambda1,lambda2) = 0.000227782
cycle = 5  norm(lambda1,lambda2) = 6.82645e-05
cycle = 6  norm(lambda1,lambda2) = 1.34748e-05
cycle = 7  norm(lambda1,lambda2) = 3.74008e-06
cycle = 8  norm(lambda1,lambda2) = 1.20021e-06
cycle = 9  norm(lambda1,lambda2) = 4.13021e-07
cycle = 10  norm(lambda1,lambda2) = 8.4982e-08
cycle = 11  norm(lambda1,lambda2) = 2.99212e-08
cycle = 12  norm(lambda1,lambda2) = 1.92209e-08
cycle = 13  norm(lambda1,lambda2) = 1.54802e-08
cycle = 14  norm(lambda1,lambda2) = 1.35677e-08
cycle = 15  norm(lambda1,lambda2) = 1.20555e-08
cycle = 16  norm(lambda1,lambda2) = 1.37011e-08
cycle = 17  norm(lambda1,lambda2) = 8.92638e-09


Step    1 : Displace = 1.007e-01/1.353e-01 (rms/max) Trust = 1.000e-01 (=) Grad = 3.580e-03/7.459e-03 (rms/max) E (change) = -115.8908546674 (-3.167e-02) Quality = 0.858
Hessian Eigenvalues: 2.30000e-02 2.30000e-02 2.54027e-02 ... 2.58375e-01 2.98781e-01 4.45091e-01


...

Step    6 : Displace = 2.582e-03/6.237e-03 (rms/max) Trust = 4.217e-03 (=) Grad = 5.440e-04/1.199e-03 (rms/max) E (change) = -115.8910168921 (-4.102e-07) Quality = 0.092
Hessian Eigenvalues: 2.30000e-02 2.30000e-02 2.54027e-02 ... 3.82408e-01 5.06628e-01 6.31020e-01


cycle = 1  norm(lambda1,lambda2) = 0.0107167
cycle = 2  norm(lambda1,lambda2) = 0.00259116
cycle = 3  norm(lambda1,lambda2) = 0.00104014
cycle = 4  norm(lambda1,lambda2) = 0.000236101
cycle = 5  norm(lambda1,lambda2) = 7.13106e-05
cycle = 6  norm(lambda1,lambda2) = 1.4155e-05
cycle = 7  norm(lambda1,lambda2) = 3.91384e-06
cycle = 8  norm(lambda1,lambda2) = 1.26146e-06
cycle = 9  norm(lambda1,lambda2) = 4.38915e-07
cycle = 10  norm(lambda1,lambda2) = 9.03288e-08
cycle = 11  norm(lambda1,lambda2) = 3.18736e-08
cycle = 12  norm(lambda1,lambda2) = 2.05066e-08
cycle = 13  norm(lambda1,lambda2) = 1.65171e-08
cycle = 14  norm(lambda1,lambda2) = 1.45056e-08
cycle = 15  norm(lambda1,lambda2) = 1.28849e-08
cycle = 16  norm(lambda1,lambda2) = 1.46486e-08
cycle = 17  norm(lambda1,lambda2) = 9.56005e-09


Step    7 : Displace = 1.340e-03/2.916e-03 (rms/max) Trust = 1.291e-03 (-) Grad = 5.378e-05/1.019e-04 (rms/max) E (change) = -115.8910190991 (-2.207e-06) Quality = 0.971
Hessian Eigenvalues: 2.30000e-02 2.30000e-02 2.54027e-02 ... 3.74729e-01 5.18948e-01 6.42379e-01


cycle = 1  norm(lambda1,lambda2) = 0.0107192
cycle = 2  norm(lambda1,lambda2) = 0.0025917
cycle = 3  norm(lambda1,lambda2) = 0.00104042
cycle = 4  norm(lambda1,lambda2) = 0.000236143
cycle = 5  norm(lambda1,lambda2) = 7.13233e-05
cycle = 6  norm(lambda1,lambda2) = 1.41576e-05
cycle = 7  norm(lambda1,lambda2) = 3.91393e-06
cycle = 8  norm(lambda1,lambda2) = 1.26147e-06
cycle = 9  norm(lambda1,lambda2) = 4.38935e-07
cycle = 10  norm(lambda1,lambda2) = 9.03452e-08
cycle = 11  norm(lambda1,lambda2) = 3.18848e-08
cycle = 12  norm(lambda1,lambda2) = 2.05101e-08
cycle = 13  norm(lambda1,lambda2) = 1.65209e-08
cycle = 14  norm(lambda1,lambda2) = 1.45098e-08
cycle = 15  norm(lambda1,lambda2) = 1.28891e-08
cycle = 16  norm(lambda1,lambda2) = 1.46521e-08
cycle = 17  norm(lambda1,lambda2) = 9.56276e-09


Step    8 : Displace = 6.981e-05/1.264e-04 (rms/max) Trust = 1.825e-03 (+) Grad = 1.156e-05/1.857e-05 (rms/max) E (change) = -115.8910191121 (-1.299e-08) Quality = 0.757
Hessian Eigenvalues: 2.30000e-02 2.30000e-02 2.54027e-02 ... 3.74729e-01 5.18948e-01 6.42379e-01
Converged! =D

    #==========================================================================#
    #| If this code has benefited your research, please support us by citing: |#
    #|                                                                        |#
    #| Wang, L.-P.; Song, C.C. (2016) "Geometry optimization made simple with |#
    #| translation and rotation coordinates", J. Chem, Phys. 144, 214108.     |#
    #| http://dx.doi.org/10.1063/1.4952956                                    |#
    #==========================================================================#
    Time elapsed since start of run_optimizer: 5.309 seconds


## 5. Optimized reference geometry

The converged CCSD(T) structure (Angstrom). This is the **reference geometry** for benchmarking the EWF results: it is stored as `propylene_ccsd_t.txt` in [`../Geom_Comparison_Tool/`](../Geom_Comparison_Tool/), where the geometries optimized with the `rdm_t` and `rdm_t_lambda` EWF assemblies are compared against it via Kabsch-aligned RMSD and per-atom deviations:

```bash
cd ../Geom_Comparison_Tool
python geom_compare.py propylene_ccsd_t.txt propylene_rdm_t.txt propylene_rdm_t_lambda.txt
```

In [5]:
print(mol_eq.tostring())

C           1.23750398       -0.19591526        0.00000000
C          -0.16060448        0.46186076        0.00000000
C          -1.32879620       -0.22823395        0.00000000
H           1.81788942        0.11352991       -0.89869507
H           1.15288542       -1.30402726        0.00000000
H           1.81788942        0.11352991        0.89869507
H          -0.17438328        1.57163593        0.00000000
H          -2.30904631        0.28387268       -0.00000000
H          -1.35260530       -1.33439206        0.00000000
